In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# 1. FILE PATHS
# =====================================
input_csv = r"E:\wenqu\2026_root\cluster\test1\root_trait_RBD.csv"
out_csv = r"E:\wenqu\2026_root\cluster\test1\cluster\root_trait_RBD.csv"

# =====================================
# 2. READ DATA
# =====================================
df = pd.read_csv(input_csv)

# =====================================
# 3. DEFINE COLUMNS
# =====================================
site_col = "site"
# Aboveground traits used for clustering
trait_cols = [
    "CWM_log10_LDMC",
    "CWM_log10_LA",
    "CWM_log10_SLA",
    "CWM_log10_PN",
    "CWM_log10_PC",
    'CWM_Ld13C',
    'CWM_Ld15N',
    'DS_relative_abund_sum',
    'ES_relative_abund_sum',
    'F_relative_abund_sum'
    
]



band_cols  = [f"b{i}" for i in range(1,123)]

C:\Users\laral\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# Coerce to numeric
df[trait_cols] = df[trait_cols].apply(pd.to_numeric, errors="coerce")
df[band_cols]  = df[band_cols].apply(pd.to_numeric, errors="coerce")

# =====================================================
# 4) KEEP ROWS THAT HAVE TRAITS + BANDS
# =====================================================
df_valid = df.dropna(subset=trait_cols + band_cols).copy()

# If you want to keep site boundary, set use_site=True
use_site = False
site_col = "site"

# number of neighbors (try 5, 10, 15)
k = 10

# Output columns
out_cols = [f"{b}_nstd" for b in band_cols]
df[out_cols] = np.nan  # fill later

# =====================================================
# 5) FUNCTION-FREE LOCAL NEIGHBORHOOD STD
# =====================================================
if not use_site:
    # ---- Global neighborhood in trait space ----
    X_trait = df_valid[trait_cols].to_numpy(float)
    Xz = StandardScaler().fit_transform(X_trait)

    nn = NearestNeighbors(n_neighbors=min(k + 1, len(df_valid)), metric="euclidean")
    nn.fit(Xz)
    neigh_idx = nn.kneighbors(Xz, return_distance=False)  # includes itself in col 0

    spec = df_valid[band_cols].to_numpy(float)

    nstd = np.zeros_like(spec)
    for i in range(spec.shape[0]):
        ids = neigh_idx[i, 1:]  # exclude itself
        if len(ids) == 0:
            nstd[i, :] = np.nan
        else:
            nstd[i, :] = np.nanstd(spec[ids, :], axis=0, ddof=0)

    df.loc[df_valid.index, out_cols] = nstd

else:
    # ---- Within-site neighborhood in trait space ----
    if site_col not in df.columns:
        raise ValueError(f"use_site=True but '{site_col}' column not found in CSV")

    for site, sub in df_valid.groupby(site_col, sort=False):
        n = len(sub)
        if n < 2:
            continue

        k_site = min(k, n - 1)  # neighbors excluding self
        X_trait = sub[trait_cols].to_numpy(float)
        Xz = StandardScaler().fit_transform(X_trait)

        nn = NearestNeighbors(n_neighbors=k_site + 1, metric="euclidean")
        nn.fit(Xz)
        neigh_idx = nn.kneighbors(Xz, return_distance=False)

        spec = sub[band_cols].to_numpy(float)

        nstd = np.zeros_like(spec)
        for i in range(n):
            ids = neigh_idx[i, 1:]  # exclude itself
            nstd[i, :] = np.nanstd(spec[ids, :], axis=0, ddof=0)

        df.loc[sub.index, out_cols] = nstd

# =====================================================
# 6) SAVE
# =====================================================
df.to_csv(out_csv, index=False)
print("Saved:", out_csv)
print("Rows with neighborhood STD:", df[out_cols[0]].notna().sum(), "out of", len(df))

Saved: E:\wenqu\2026_root\cluster\test1\cluster\root_trait_RBD.csv
Rows with neighborhood STD: 82 out of 82


C:\Users\laral\AppData\Local\Temp\ipykernel_10428\112510550.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[out_cols] = np.nan  # fill later
C:\Users\laral\AppData\Local\Temp\ipykernel_10428\112510550.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[out_cols] = np.nan  # fill later
C:\Users\laral\AppData\Local\Temp\ipykernel_10428\112510550.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all c

In [4]:
import pandas as pd

df = pd.read_csv(R"E:\wenqu\2026_root\cluster\test2\root_trait_RBD.csv")

bands = [f"b{i}" for i in range(1,123)]

df["spec_mean"] = df[bands].mean(axis=1)
df["spec_std"] = df[bands].std(axis=1)

In [6]:
df

,site,RDMC_A,SRL,RTD,RD,SRA,RBD,RLD,RCC,Rd13C,...,CWM_log10_PN,CWM_log10_LDMC,CWM_log10_PC,DS_relative_abund_sum,ES_relative_abund_sum,F_relative_abund_sum,G_relative_abund_sum,M_relative_abund_sum,spec_mean,spec_std
0,S2A_Q03,336.738703,0.000000,0.000000,0.000000,0.000000,0.002622,0.000000,423.179198,-28.593295,...,1.017168,2.892741,2.631377,50.000000,25.000000,25.000000,0.000000,0.000000,0.136741,0.107610
1,S2A_Q04,353.271028,0.000000,0.000000,0.000000,0.000000,0.002062,0.000000,456.907537,-28.606056,...,1.352571,2.614347,2.663195,78.651685,4.494382,16.853933,0.000000,0.000000,0.131029,0.100460
2,S2A_Q05,175.438596,0.000000,0.000000,0.000000,0.000000,0.003750,0.000000,457.454297,-27.445805,...,0.839583,2.664934,2.629382,15.837104,7.239819,76.923077,0.000000,0.000000,0.054771,0.038295
3,S2A_Q07,596.412556,40.710812,0.343669,0.301668,385.201504,0.001307,2.671432,466.883585,-28.566239,...,1.030636,2.817267,2.650235,53.571429,30.059524,16.369048,0.000000,0.000000,0.083248,0.057927
4,S2A_Q08,372.549020,96.196211,0.253333,0.228576,692.194737,0.000355,0.901767,439.500101,-28.717293,...,1.205865,2.619883,2.600606,19.607843,1.960784,78.431373,0.000000,0.000000,0.058835,0.043755
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,S7_Q20,257.772601,136.121754,0.584422,0.126511,541.220120,0.004291,6.942844,424.600000,-28.292000,...,0.970116,2.422870,2.500887,18.421053,0.000000,2.631579,77.192982,1.754386,0.213369,0.148701
125,S7_Q21,263.733144,190.151161,0.504828,0.115169,687.338798,0.003750,6.104361,411.490000,-27.654000,...,1.147319,2.502558,2.521443,12.658228,0.000000,1.265823,60.759494,25.316456,0.114316,0.086946
126,S7_Q22,224.401607,193.358806,0.440957,0.122201,741.851628,0.006126,9.586216,436.110000,-28.360000,...,1.339741,2.532167,2.646161,37.500000,0.000000,0.000000,62.500000,0.000000,0.195218,0.135987
127,S7_Q23,212.132938,165.517248,0.620833,0.111313,578.797315,0.008013,16.223739,432.800000,-28.113000,...,1.200604,2.540229,2.627366,0.000000,0.000000,0.000000,100.000000,0.000000,0.042651,0.028573


In [7]:
out_csv = r"E:\wenqu\2026_root\cluster\test2\CLUSTER\root_trait_RBD.csv"
df.to_csv(out_csv, index=False)

In [25]:
import pandas as pd

# =========================
# 1. READ CSV
# =========================
file = r"D:\wenqu\2024_data\SVC\update4\spectral_trait_mean_sla.csv"
df = pd.read_csv(file)

# =========================
# 2. DEFINE BAND COLUMNS
# =========================
bands = [f"b{i}" for i in range(1,123)]

# =========================
# 3. CALCULATE SPECTRAL DERIVATIVES
# =========================
for i in range(1,122):
    
    b1 = f"b{i}"
    b2 = f"b{i+1}"
    
    df[f"d{i}"] = df[b2] - df[b1]

# =========================
# 4. SAVE OUTPUT
# =========================
out_file = r"D:\wenqu\2024_data\SVC\update4\diff\spectral_trait_mean_sla.csv"
df.to_csv(out_file, index=False)

print("Derivative bands created:", [f"d{i}" for i in range(1,122)])
print("Saved file:", out_file)

Derivative bands created: ['d1', 'd2', 'd3', 'd4', 'd5', 'd6', 'd7', 'd8', 'd9', 'd10', 'd11', 'd12', 'd13', 'd14', 'd15', 'd16', 'd17', 'd18', 'd19', 'd20', 'd21', 'd22', 'd23', 'd24', 'd25', 'd26', 'd27', 'd28', 'd29', 'd30', 'd31', 'd32', 'd33', 'd34', 'd35', 'd36', 'd37', 'd38', 'd39', 'd40', 'd41', 'd42', 'd43', 'd44', 'd45', 'd46', 'd47', 'd48', 'd49', 'd50', 'd51', 'd52', 'd53', 'd54', 'd55', 'd56', 'd57', 'd58', 'd59', 'd60', 'd61', 'd62', 'd63', 'd64', 'd65', 'd66', 'd67', 'd68', 'd69', 'd70', 'd71', 'd72', 'd73', 'd74', 'd75', 'd76', 'd77', 'd78', 'd79', 'd80', 'd81', 'd82', 'd83', 'd84', 'd85', 'd86', 'd87', 'd88', 'd89', 'd90', 'd91', 'd92', 'd93', 'd94', 'd95', 'd96', 'd97', 'd98', 'd99', 'd100', 'd101', 'd102', 'd103', 'd104', 'd105', 'd106', 'd107', 'd108', 'd109', 'd110', 'd111', 'd112', 'd113', 'd114', 'd115', 'd116', 'd117', 'd118', 'd119', 'd120', 'd121']
Saved file: D:\wenqu\2024_data\SVC\update4\diff\spectral_trait_mean_sla.csv


C:\Users\laral\AppData\Local\Temp\ipykernel_10428\3949382246.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"d{i}"] = df[b2] - df[b1]
C:\Users\laral\AppData\Local\Temp\ipykernel_10428\3949382246.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"d{i}"] = df[b2] - df[b1]
C:\Users\laral\AppData\Local\Temp\ipykernel_10428\3949382246.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at